# DayTracker – starter analysis

1. In the app: **Settings → Data → Export all data**, save the `.zip` (Files / iCloud Drive / email).
2. Put it in `analysis/data/` (ignored by git – your data never gets committed).
3. Run all cells. `dt.load()` picks the newest export.

Days run 04:00–04:00 (D14), so late-night entries count for the evening before.

In [ ]:
import daytracker as dt
import matplotlib.pyplot as plt

ex = dt.load()  # or dt.load('data/DayTracker-2026-11-10-2130.zip')
print(f"{ex.source.name}: {len(ex.entries)} entries, "
      f"{ex.entries['day'].min():%d %b} – {ex.entries['day'].max():%d %b}")
ex.entries.tail()

## Day by day
Counters are daily totals, medicines are doses taken, ratings are the day's average.

In [ ]:
daily = dt.daily(ex)
daily.tail(14).round(1)

In [ ]:
ratings = [c for c in daily.columns if c in set(ex.trackers.loc[ex.trackers['kind'] == 'scale', 'name'])]
ax = daily[ratings].plot(marker='o', figsize=(10, 4), ylim=(1, 10), title='Daily average ratings')
ax.grid(alpha=0.3)

## Ratings by time of day

In [ ]:
dt.plot_by_hour(ex);

## Weekdays

In [ ]:
dt.by_weekday(ex).round(1)

## Caffeine timing vs. evening energy
`last_coffee_hour` counts from midnight, so 25.5 means 01:30 the night after.

In [ ]:
coffee = dt.last_coffee_vs_evening(ex)
ax = coffee.plot.scatter(x='last_coffee_hour', y='evening_energy', figsize=(6, 4),
                         title='Last coffee vs. evening energy')
ax.grid(alpha=0.3)
coffee.dropna().corr().round(2)

## Water vs. energy

In [ ]:
if {'Water', 'Energy'} <= set(daily.columns):
    ax = daily.plot.scatter(x='Water', y='Energy', figsize=(6, 4), title='Water vs. average energy')
    ax.grid(alpha=0.3)
    print(daily[['Water', 'Energy']].dropna().corr().round(2))

## Medication adherence
`expected_slots` assumes today's reminder times applied every day in the export.

In [ ]:
dt.adherence(ex)

## Check-ins

In [ ]:
dt.checkin_summary(ex)